# Phase 1: Hybrid Legal Language Model

## 1. Check GPU

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 3. Hugging Face login


In [2]:
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
login(token=HF_TOKEN)


## 4. Load Tokenizer
Reuse the byte-level BPE implementation. The first four special tokens stay in their original order; `<mask>` is added as a fifth special token for corruption. The tokenizer is trained only on the training-document split. Do not reuse checkpoints with a different vocabulary mapping.


In [3]:
from collections import defaultdict, Counter
import heapq
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>', '<mask>']

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        self.symbol_to_bytes = {}

    def word_to_symbols(self, word):
        byte_seq = word.encode('utf-8')
        return [str(b) for b in byte_seq] + ['</w>']

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print('Building word frequencies...')
        for text in corpus:
            if not isinstance(text, str):
                continue
            word_counter.update(text.split())
        print(f'Unique whitespace-separated words: {len(word_counter):,}')
        word_freqs = {}
        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = word_freqs.get(symbols, 0) + freq
        return word_freqs

    @staticmethod
    def get_pairs(symbols):
        return {(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)}

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[symbols[i], symbols[i + 1]] += 1
        return counts

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and (symbols[i + 1] == pair[1]):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        return tuple(new_symbols)

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = self.specials + self.byte_vocab + ['</w>']
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes['</w>'] = b''
        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs
        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)
        print('Building initial pair statistics...')
        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)
        print(f'Initial unique pairs: {len(pair_counts):,}')
        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap, (-count, pair))
        target_merges = max(0, self.vocab_size - len(vocab))
        print(f'Target merges: {target_merges:,}')
        while len(vocab) < self.vocab_size and heap:
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair, 0)
                if current_count == heap_count and current_count > 0:
                    break
            else:
                break
            max_freq = pair_counts.get(pair, 0)
            if max_freq <= 1:
                break
            merged_symbol = f'<bpe_{len(self.merges)}>'
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes
            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1, merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)
            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                if pair not in self.get_pairs(old_symbols):
                    continue
                old_counts = self.pair_counts_in_word(old_symbols)
                new_symbols = self.merge_symbols(old_symbols, pair, merged_symbol)
                if new_symbols == old_symbols:
                    continue
                new_counts = self.pair_counts_in_word(new_symbols)
                words[word_id] = new_symbols
                all_pairs = set(old_counts) | set(new_counts)
                for p in all_pairs:
                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (new_n - old_n) * freq
                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)
            pair_to_words[pair].clear()
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap, (-count, p))
        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {symbol: i for i, symbol in enumerate(vocab)}
        self.itos = {i: symbol for symbol, i in self.stoi.items()}
        print('\nTokenizer training complete.')
        print('Final vocabulary size:', self.vocab_size)
        print('Number of merges:', len(self.merges))

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float('inf')
            best_merged_symbol = None
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol
            if best_pair is None:
                break
            symbols = list(self.merge_symbols(tuple(symbols), best_pair, best_merged_symbol))
        return symbols

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos:
            ids.append(self.stoi['<s>'])
        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(self.stoi.get(symbol, self.stoi['<unk>']))
        if add_eos:
            ids.append(self.stoi['</s>'])
        return ids

    def decode(self, ids):
        words = []
        current_bytes = bytearray()
        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b'')
            current_bytes.extend(byte_value)
            if ends_word:
                try:
                    word = bytes(current_bytes).decode('utf-8')
                    words.append(word)
                except UnicodeDecodeError:
                    words.append('<unk>')
                current_bytes = bytearray()
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode('utf-8'))
            except UnicodeDecodeError:
                words.append('<unk>')
        return ' '.join(words)

    def _symbol_ends_word(self, symbol):
        if symbol == '</w>':
            return True
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(pair[1])
        return False

    def get_vocab_size(self):
        return len(self.vocab)


## 9. Define The Model
Each block has gated multi-head attention and a GEGLU feed-forward layer, with residual connections. The same weights process both objectives. `causal=False` gives bidirectional encoder attention. `causal=True` gives left-to-right attention. KV caching is allowed only in causal evaluation, because adding a token in bidirectional mode would change earlier representations.


In [4]:
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class HybridConfig:
    vocab_size: int
    pad_id: int
    mask_id: int
    d_model: int = 512
    n_layers: int = 12
    n_heads: int = 8
    d_ff: int = 1664
    max_seq_len: int = 512
    dropout: float = 0.1


class GatedAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.gate = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.output_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, key_valid, causal=True, past=None, use_cache=False):
        b, t, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        past_len = 0 if past is None else past[0].size(2)
        if past is not None:
            k = torch.cat((past[0], k), dim=2)
            v = torch.cat((past[1], v), dim=2)

        allowed = key_valid[:, None, None, :]
        if causal:
            query_positions = torch.arange(past_len, past_len + t, device=x.device)
            key_positions = torch.arange(k.size(2), device=x.device)
            allowed = allowed & (key_positions[None, :] <= query_positions[:, None])[None, None]

        # A fully padded query row gets a harmless fallback key to avoid NaNs.
        first_key = torch.arange(k.size(2), device=x.device).eq(0)
        allowed = allowed | (~allowed.any(dim=-1, keepdim=True) & first_key)
        # Explicit mask also handles offset queries when a KV cache is present.
        context = F.scaled_dot_product_attention(
            q, k, v, attn_mask=allowed,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=False,
        )
        context = context.transpose(1, 2).contiguous().view(b, t, d)
        # Paper Section 2.2: GEGLU gate and non-affine output LayerNorm.
        context = context * F.gelu(self.gate(x))
        output = self.out(self.output_norm(context))
        return output, (k, v) if use_cache else None


In [5]:
class GatedFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.value = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.output_norm = nn.LayerNorm(cfg.d_ff, elementwise_affine=False)
        self.out = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        hidden = self.value(x) * F.gelu(self.gate(x))
        return self.out(self.output_norm(hidden))


class HybridBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.norm2 = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.attn = GatedAttention(cfg)
        self.ffn = GatedFeedForward(cfg)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, valid, causal, past=None, use_cache=False):
        attention, kv = self.attn(self.norm1(x), valid, causal, past, use_cache)
        x = x + self.dropout(attention)
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x, kv


class GPTBertLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        self.embedding_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([HybridBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model, elementwise_affine=False)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self._initialize)
        self.lm_head.weight = self.tok_emb.weight
        # Smaller residual projections keep a deep stack stable at initialization.
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
            nn.init.normal_(block.ffn.out.weight, std=0.02 / math.sqrt(2 * cfg.n_layers))
        with torch.no_grad():
            self.tok_emb.weight[cfg.pad_id].zero_()

    def _initialize(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, ids, causal=True, cache=None, use_cache=False, last_only=False):
        if (cache is not None or use_cache) and (not causal or self.training):
            raise ValueError('KV caching is only used in causal evaluation mode.')
        offset = 0 if cache is None else cache['length']
        if offset + ids.size(1) > self.cfg.max_seq_len:
            raise ValueError('Context is full: rebuild from the most recent tokens.')
        positions = torch.arange(offset, offset + ids.size(1), device=ids.device)
        x = self.dropout(self.embedding_norm(self.tok_emb(ids) + self.pos_emb(positions)))
        valid = ids.ne(self.cfg.pad_id)
        if cache is not None:
            valid = torch.cat((cache['valid'], valid), dim=1)
        updated = []
        for i, block in enumerate(self.blocks):
            past = None if cache is None else cache['kv'][i]
            x, kv = block(x, valid, causal, past, use_cache)
            if use_cache:
                updated.append(kv)
        logits = self.lm_head(self.final_norm(x[:, -1:] if last_only else x))
        if use_cache:
            return logits, dict(kv=updated, valid=valid, length=offset + ids.size(1))
        return logits


# **Phase 7**

In [6]:
import gc
import json
import math
import random
import shutil
import zipfile

from pathlib import Path
from functools import lru_cache
from types import MethodType

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.nn.utils.rnn import pad_sequence
from torch.utils.checkpoint import checkpoint

from sklearn.feature_extraction.text import TfidfVectorizer
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient


HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

SEED = 42

from torch.utils.checkpoint import checkpoint as checkpoint_block

MODEL_DIR = Path(
    "/kaggle/input/datasets/trishalreddybokka/model3-for-phase7-files"
)

OUTPUT_DIR = Path("/kaggle/working/phase7_model3_lora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

LEARNING_RATE = 1e-4
PROJECTION_DIM = 256

# Complete texts are available for retrieval and reranking.
PHASE7_CAUSAL = False

# Model 3 has a 512-token positional embedding table.
RETRIEVAL_LENGTH = 512
PAIR_LENGTH = 512

MAX_CHUNKS = 8
CHUNK_OVERLAP = 64
PASSAGE_OVERLAP = 64
PASSAGE_VIEWS = 2

QUERY_BATCH_SIZE = 8
CHUNK_BATCH_SIZE = 8
PAIR_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16

RETRIEVER_CHECKPOINTING = True
RERANKER_CHECKPOINTING = False

POSITIVES = 2
HARD_NEGATIVES = 2
RANDOM_NEGATIVES = 1

RETRIEVER_EPOCHS = 5
RERANKER_EPOCHS = 3
PATIENCE = 2

RETRIEVE_TOP_K = 100
RERANK_TOP_K = 50

NEURAL_WEIGHTS = [0.0, 0.25, 0.5, 0.75, 1.0]
FUSION_WEIGHTS = [0.0, 0.25, 0.5, 0.75, 1.0]

# Reciprocal rank fusion
RRF_K = 60

# Contrastive retrieval loss
TEMPERATURE = 0.07

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

use_amp = device.type == "cuda"

amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)


def amp_context():
    return torch.autocast(
        device_type=device.type,
        dtype=amp_dtype,
        enabled=use_amp,
    )


def load_json(path):
    with open(path, encoding="utf-8") as file:
        return json.load(file)

In [7]:
def load_model3():
    saved = torch.load(
        MODEL_DIR / "model.pt",
        map_location="cpu",
        weights_only=True,
    )

    cfg = HybridConfig(**saved["model_config"])
    model = GPTBertLM(cfg)

    model.load_state_dict(
        saved["model_state_dict"],
        strict=True,
    )

    return model

state = load_json(MODEL_DIR / "tokenizer.json")

tokenizer = BytelevelBPE(state["vocab_size"])
tokenizer.specials = state["specials"]
tokenizer.vocab = state["vocab"]
tokenizer.vocab_size = len(tokenizer.vocab)

tokenizer.stoi = {
    symbol: index
    for index, symbol in enumerate(tokenizer.vocab)
}

tokenizer.itos = {
    index: symbol
    for symbol, index in tokenizer.stoi.items()
}

tokenizer.merges = [
    tuple(pair) for pair in state["merges"]
]

tokenizer.merge_ranks = {
    tuple(row["pair"]): (
        row["rank"],
        row["merged_symbol"],
    )
    for row in state["merge_ranks"]
}

tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}

tokenizer.apply_bpe = lru_cache(
    maxsize=100000
)(tokenizer.apply_bpe)

del state

PAD_ID = tokenizer.stoi["<pad>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]


def pack_sequences(sequences):
    lengths = torch.tensor(
        [len(sequence) for sequence in sequences],
        device=device,
    )

    ids = pad_sequence(
        [
            torch.tensor(sequence, dtype=torch.long)
            for sequence in sequences
        ],
        batch_first=True,
        padding_value=PAD_ID,
    ).to(device)

    return ids, lengths

In [8]:
def read_cases(path, labeled=False):
    content = load_json(path)

    if isinstance(content, dict):
        items = content.items()
    else:
        items = [
            (
                row.get(
                    "id",
                    row.get("query_id", row.get("case_id")),
                ),
                row,
            )
            for row in content
        ]

    records = {}

    for case_id, row in items:
        if case_id is None:
            raise ValueError("Missing case ID.")

        if not isinstance(row, dict):
            row = {"text": row}

        text = row["text"]

        if isinstance(text, list):
            text = " ".join(
                part if isinstance(part, str) else part["text"]
                for part in text
            )

        records[str(case_id)] = {"text": text}

        if labeled:
            records[str(case_id)]["relevant_candidates"] = {
                str(candidate_id)
                for candidate_id in row["relevant_candidates"]
            }

    return records


data = {}
tokens = {}

for split in ["train", "dev", "test"]:
    records = []

    for filename in [
        f"{split}.json",
        f"{split}_candidates.json",
    ]:
        path = hf_hub_download(
            repo_id="Exploration-Lab/CS-781-Capstone",
            repo_type="dataset",
            filename=f"Phase_7_PCR/{split.title()}/{filename}",
            token=HF_TOKEN,
        )

        labeled = (
            split != "test"
            and filename == f"{split}.json"
        )

        records.append(
            read_cases(path, labeled=labeled)
        )

    # Each split contains (queries, candidates).
    data[split] = tuple(records)

    tokens[split] = tuple(
        {
            case_id: tokenizer.encode(row["text"])
            for case_id, row in group.items()
        }
        for group in records
    )

    print(
        split,
        len(records[0]), "queries,",
        len(records[1]), "candidates",
    )


train_queries, train_candidates = data["train"]
dev_queries, dev_candidates = data["dev"]
test_queries, test_candidates = data["test"]

for queries, candidates in [data["train"], data["dev"]]:
    for row in queries.values():
        assert row["relevant_candidates"] <= set(candidates)

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

train 827 queries, 4320 candidates


dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (3 outputs left out)

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test 237 queries, 1727 candidates


In [9]:
class LoRAQKV(nn.Module):
    """Independent LoRA updates for Q and V inside fused QKV."""

    def __init__(self, linear):
        super().__init__()

        self.base = linear
        self.base.requires_grad_(False)

        width = linear.out_features // 3

        options = {
            "device": linear.weight.device,
            "dtype": linear.weight.dtype,
        }

        self.A_q = nn.Linear(
            linear.in_features,
            LORA_R,
            bias=False,
            **options,
        )

        self.B_q = nn.Linear(
            LORA_R,
            width,
            bias=False,
            **options,
        )

        self.A_v = nn.Linear(
            linear.in_features,
            LORA_R,
            bias=False,
            **options,
        )

        self.B_v = nn.Linear(
            LORA_R,
            width,
            bias=False,
            **options,
        )

        self.dropout = nn.Dropout(LORA_DROPOUT)
        self.scale = LORA_ALPHA / LORA_R

        for A, B in [
            (self.A_q, self.B_q),
            (self.A_v, self.B_v),
        ]:
            nn.init.kaiming_uniform_(
                A.weight,
                a=math.sqrt(5),
            )
            nn.init.zeros_(B.weight)

    def forward(self, x):
        q, k, v = self.base(x).chunk(3, dim=-1)

        dropped = self.dropout(x)

        q = q + self.scale * self.B_q(
            self.A_q(dropped)
        )

        v = v + self.scale * self.B_v(
            self.A_v(dropped)
        )

        return torch.cat([q, k, v], dim=-1)


class Phase7Model(nn.Module):
    def __init__(self, task):
        super().__init__()

        base = load_model3()

        self.cfg = base.cfg
        self.task = task
        self.causal = PHASE7_CAUSAL

        self.tok_emb = base.tok_emb
        self.pos_emb = base.pos_emb
        self.embedding_norm = base.embedding_norm
        self.dropout = base.dropout
        self.blocks = base.blocks
        self.final_norm = base.final_norm

        # Freeze original Model 3 parameters.
        self.requires_grad_(False)

        for block in self.blocks:
            block.attn.qkv = LoRAQKV(block.attn.qkv)

        width = PROJECTION_DIM if task == "retriever" else 1

        self.head = nn.Linear(
            self.cfg.d_model,
            width,
            bias=False,
        )

    def forward(self, ids, lengths):
        length = ids.shape[1]

        if length > self.cfg.max_seq_len:
            raise ValueError(
                "Input exceeds Model 3's positional embedding limit."
            )

        positions = torch.arange(
            length,
            device=ids.device,
        )

        x = self.tok_emb(ids) + self.pos_emb(positions)
        x = self.dropout(self.embedding_norm(x))

        valid = ids.ne(self.cfg.pad_id)

        if self.task == "retriever":
            use_checkpoint = RETRIEVER_CHECKPOINTING
        else:
            use_checkpoint = RERANKER_CHECKPOINTING

        for block in self.blocks:
            if self.training and use_checkpoint:
                x, _ = checkpoint_block(
                    block,
                    x,
                    valid,
                    self.causal,
                    use_reentrant=False,
                )
            else:
                x, _ = block(
                    x,
                    valid,
                    self.causal,
                )

        rows = torch.arange(
            len(ids),
            device=ids.device,
        )

        # Representation at the final EOS, excluding padding.
        return self.final_norm(
            x[rows, lengths - 1]
        )


def adapter_settings():
    return {
        "rank": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "projection_dim": PROJECTION_DIM,
        "causal": PHASE7_CAUSAL,
        "targets": "q_v",
        "pool": "last_eos",
    }


def save_model(model, path, epoch, result):
    saved = {
        "method": "model3_lora",
        "task": model.task,
        "epoch": epoch,
        "result": result,
        "settings": adapter_settings(),

        "trainable_state": {
            name: parameter.detach().cpu().clone()
            for name, parameter in model.named_parameters()
            if parameter.requires_grad
        },
    }

    temporary = path.with_suffix(".tmp")

    torch.save(saved, temporary)
    temporary.replace(path)


def load_phase7(path):
    saved = torch.load(
        path,
        map_location="cpu",
        weights_only=True,
    )

    if (
        saved.get("method") != "model3_lora"
        or saved["settings"] != adapter_settings()
    ):
        raise ValueError(
            "Checkpoint must match Model 3 LoRA and these settings."
        )

    model = Phase7Model(saved["task"])

    expected = {
        name
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }

    if set(saved["trainable_state"]) != expected:
        raise ValueError(
            "Checkpoint trainable parameters do not match."
        )

    model.load_state_dict(
        saved["trainable_state"],
        strict=False,
    )

    return model.to(device).eval(), saved


retriever = Phase7Model("retriever").to(device)

assert retriever.cfg.vocab_size == tokenizer.vocab_size
assert retriever.cfg.pad_id == PAD_ID
assert retriever.cfg.mask_id == tokenizer.stoi["<mask>"]

RETRIEVAL_LENGTH = min(
    RETRIEVAL_LENGTH,
    retriever.cfg.max_seq_len,
)

PAIR_LENGTH = min(
    PAIR_LENGTH,
    retriever.cfg.max_seq_len,
)

# Format: BOS + query + EOS + candidate + EOS
PASSAGE_TOKENS = (PAIR_LENGTH - 3) // 2

assert 0 <= CHUNK_OVERLAP < RETRIEVAL_LENGTH - 2
assert 0 <= PASSAGE_OVERLAP < PASSAGE_TOKENS

In [10]:
def split_tokens(ids, size, overlap):
    parts = []

    for start in range(0, len(ids), size - overlap):
        parts.append(ids[start:start + size])

        if start + size >= len(ids):
            break

    return parts or [[]]


def retrieval_chunks(ids):
    parts = split_tokens(
        ids,
        RETRIEVAL_LENGTH - 2,
        CHUNK_OVERLAP,
    )

    if len(parts) > MAX_CHUNKS:
        positions = np.linspace(
            0,
            len(parts) - 1,
            MAX_CHUNKS,
            dtype=int,
        )

        parts = [parts[i] for i in positions]

    return [
        [BOS_ID] + part + [EOS_ID]
        for part in parts
    ]


chunks = {}

for split in tokens:
    chunks[split] = tuple(
        {
            case_id: retrieval_chunks(ids)
            for case_id, ids in group.items()
        }
        for group in tokens[split]
    )


def lexical_rankings(split):
    queries, candidates = data[split]
    candidate_ids = list(candidates)

    tfidf = TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True,
        max_features=60000,
        dtype=np.float32,
    )

    matrix = tfidf.fit_transform(
        [candidates[c]["text"] for c in candidate_ids]
    )

    rankings = {}

    for query_id, row in queries.items():
        query_vector = tfidf.transform([row["text"]])

        scores = (
            query_vector @ matrix.T
        ).toarray().ravel()

        order = np.argsort(
            -scores,
            kind="stable",
        )[:RETRIEVE_TOP_K]

        rankings[query_id] = [
            candidate_ids[i] for i in order
        ]

    return rankings


lexical = {
    split: lexical_rankings(split)
    for split in ["train", "dev"]
}


def encode_documents(model, documents):
    sequences = []
    owners = []

    for owner, document in enumerate(documents):
        sequences.extend(document)
        owners.extend([owner] * len(document))

    vectors = []

    for start in range(0, len(sequences), CHUNK_BATCH_SIZE):
        batch = sequences[start:start + CHUNK_BATCH_SIZE]

        vectors.append(
            model(*pack_sequences(batch)).float()
        )

    vectors = torch.cat(vectors)

    owners = torch.tensor(
        owners,
        device=device,
    )

    totals = vectors.new_zeros(
        len(documents),
        vectors.shape[1],
    ).index_add(
        0,
        owners,
        vectors,
    )

    counts = torch.bincount(
        owners,
        minlength=len(documents),
    ).unsqueeze(1)

    document_vectors = totals / counts

    return F.normalize(
        model.head(document_vectors).float(),
        dim=-1,
    )


@torch.inference_mode()
def make_index(model, records):
    model.eval()

    ids = list(records)
    vectors = []

    for start in range(0, len(ids), EVAL_BATCH_SIZE):
        batch_ids = ids[start:start + EVAL_BATCH_SIZE]

        with amp_context():
            batch_vectors = encode_documents(
                model,
                [records[case_id] for case_id in batch_ids],
            )

        vectors.append(batch_vectors.cpu())

    return ids, torch.cat(vectors)


def neural_rankings(model, split):
    query_ids, query_vectors = make_index(
        model,
        chunks[split][0],
    )

    candidate_ids, candidate_vectors = make_index(
        model,
        chunks[split][1],
    )

    scores = query_vectors @ candidate_vectors.T
    rankings = {}

    for query_id, row in zip(query_ids, scores):
        order = torch.argsort(
            row,
            descending=True,
            stable=True,
        ).tolist()[:RETRIEVE_TOP_K]

        rankings[query_id] = [
            candidate_ids[i] for i in order
        ]

    return rankings


def fuse_rankings(first, second, weight):
    fused = {}

    for query_id in first:
        scores = {}

        for ranking, source_weight in [
            (first[query_id], weight),
            (second[query_id], 1.0 - weight),
        ]:
            for rank, candidate_id in enumerate(ranking, 1):
                scores[candidate_id] = (
                    scores.get(candidate_id, 0.0)
                    + source_weight / (RRF_K + rank)
                )

        fused[query_id] = sorted(
            scores,
            key=lambda c: (-scores[c], c),
        )

    return fused


def retrieval_scores(rankings, queries):
    assert set(rankings) == set(queries)

    hits = sum(
        len(
            set(rankings[query_id][:7])
            & row["relevant_candidates"]
        )
        for query_id, row in queries.items()
    )

    gold = sum(
        len(row["relevant_candidates"])
        for row in queries.values()
    )

    predicted = 7 * len(queries)

    return {
        "precision": hits / predicted,
        "recall": hits / max(gold, 1),
        "f1": 2 * hits / (predicted + gold),
    }

In [11]:
training_ids = [
    query_id
    for query_id, row in train_queries.items()
    if (
        0 < len(row["relevant_candidates"])
        < len(train_candidates)
    )
]

train_candidate_ids = list(train_candidates)

assert training_ids


def sample_candidates(query_id, rankings):
    gold = train_queries[query_id]["relevant_candidates"]

    positives = random.sample(
        sorted(gold),
        min(POSITIVES, len(gold)),
    )

    hard_pool = [
        candidate_id
        for candidate_id in rankings[query_id]
        if candidate_id not in gold
    ][:50]

    negatives = random.sample(
        hard_pool,
        min(HARD_NEGATIVES, len(hard_pool)),
    )

    other = [
        candidate_id
        for candidate_id in train_candidate_ids
        if candidate_id not in gold
        and candidate_id not in negatives
    ]

    negatives += random.sample(
        other,
        min(RANDOM_NEGATIVES, len(other)),
    )

    assert negatives

    return positives + negatives, len(positives)


def retriever_loss(model, query_ids):
    candidates = []

    for query_id in query_ids:
        sampled, _ = sample_candidates(
            query_id,
            lexical["train"],
        )
        candidates.extend(sampled)

    candidates = list(dict.fromkeys(candidates))

    query_vectors = encode_documents(
        model,
        [chunks["train"][0][q] for q in query_ids],
    )

    candidate_vectors = encode_documents(
        model,
        [chunks["train"][1][c] for c in candidates],
    )

    scores = (
        query_vectors @ candidate_vectors.T
    ) / TEMPERATURE

    positive_mask = torch.tensor(
        [
            [
                candidate_id
                in train_queries[query_id]["relevant_candidates"]
                for candidate_id in candidates
            ]
            for query_id in query_ids
        ],
        device=device,
        dtype=torch.bool,
    )

    log_probs = F.log_softmax(
        scores.float(),
        dim=-1,
    )

    loss_per_query = -(
        log_probs * positive_mask
    ).sum(1) / positive_mask.sum(1)

    return loss_per_query.mean()


def evaluate_retriever(model):
    rankings = neural_rankings(model, "dev")

    return retrieval_scores(
        rankings,
        dev_queries,
    )

def train_stage(
    model,
    loss_function,
    evaluate,
    epochs,
    filename,
    baseline=None,
):
    path = OUTPUT_DIR / filename

    if path.exists():
        raise FileExistsError(
            f"{path} already exists. "
            "Use a new output directory for a fresh experiment."
        )

    parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        parameters,
        lr=LEARNING_RATE,
        weight_decay=0.01,
        fused=(device.type == "cuda"),
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp and amp_dtype == torch.float16,
    )

    best_f1 = -1.0 if baseline is None else baseline["f1"]
    bad_epochs = 0
    history = []

    if baseline is not None:
        save_model(model, path, 0, baseline)

    for epoch in range(1, epochs + 1):
        model.train()

        order = training_ids.copy()
        random.shuffle(order)

        total_loss = 0.0
        batches = math.ceil(len(order) / QUERY_BATCH_SIZE)

        for step, start in enumerate(
            range(0, len(order), QUERY_BATCH_SIZE),
            start=1,
        ):
            query_ids = order[start:start + QUERY_BATCH_SIZE]

            optimizer.zero_grad(set_to_none=True)

            # Reranker queries have independent losses.
            # Keep retriever queries together for shared candidates.
            if model.task == "reranker":
                micro_batches = [[q] for q in query_ids]
            else:
                micro_batches = [query_ids]

            batch_loss = torch.zeros((), device=device)

            for micro_ids in micro_batches:
                with amp_context():
                    loss = loss_function(model, micro_ids)

                weight = len(micro_ids) / len(query_ids)
                scaler.scale(loss * weight).backward()

                batch_loss += loss.detach().float() * weight

            if not torch.isfinite(batch_loss):
                raise RuntimeError("Non-finite training loss.")

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                parameters,
                max_norm=1.0,
            )

            scaler.step(optimizer)
            scaler.update()

            loss_value = batch_loss.item()
            total_loss += loss_value * len(query_ids)

            if step % 100 == 0 or step == batches:
                print(
                    f"{model.task} | Epoch {epoch} | "
                    f"Batch {step}/{batches} | "
                    f"Loss: {loss_value:.4f}",
                    flush=True,
                )

        model.eval()

        with torch.inference_mode():
            result = evaluate(model)

        record = {
            "epoch": epoch,
            "train_loss": total_loss / len(order),
            **result,
        }

        history.append(record)
        print(record, flush=True)

        if result["f1"] > best_f1:
            best_f1 = result["f1"]
            bad_epochs = 0

            save_model(model, path, epoch, result)
            print("Saved best checkpoint.", flush=True)

        else:
            bad_epochs += 1

            if bad_epochs >= PATIENCE:
                break

    return history

retriever_history = train_stage(
    retriever,
    retriever_loss,
    evaluate_retriever,
    RETRIEVER_EPOCHS,
    "best_retriever.pt",
)

del retriever
gc.collect()

if use_amp:
    torch.cuda.empty_cache()

retriever, retriever_saved = load_phase7(
    OUTPUT_DIR / "best_retriever.pt"
)

retriever | Epoch 1 | Batch 100/103 | Loss: 2.9496
retriever | Epoch 1 | Batch 103/103 | Loss: 2.7023
{'epoch': 1, 'train_loss': 2.616723482582531, 'precision': 0.24455205811138014, 'recall': 0.20760534429599178, 'f1': 0.2245692051139522}
Saved best checkpoint.
retriever | Epoch 2 | Batch 100/103 | Loss: 2.8184
retriever | Epoch 2 | Batch 103/103 | Loss: 2.6005
{'epoch': 2, 'train_loss': 2.327592745842811, 'precision': 0.24818401937046006, 'recall': 0.21068859198355602, 'f1': 0.22790439132851584}
Saved best checkpoint.
retriever | Epoch 3 | Batch 100/103 | Loss: 2.6447
retriever | Epoch 3 | Batch 103/103 | Loss: 3.7828
{'epoch': 3, 'train_loss': 2.3285434327773396, 'precision': 0.2687651331719128, 'recall': 0.22816032887975335, 'f1': 0.24680377987770985}
Saved best checkpoint.
retriever | Epoch 4 | Batch 100/103 | Loss: 2.4836
retriever | Epoch 4 | Batch 103/103 | Loss: 0.6668
{'epoch': 4, 'train_loss': 2.2252814272188353, 'precision': 0.28329297820823246, 'recall': 0.24049331963001028

In [12]:
dev_neural = neural_rankings(
    retriever,
    "dev",
)

results = []

for weight in NEURAL_WEIGHTS:
    rankings = fuse_rankings(
        dev_neural,
        lexical["dev"],
        weight,
    )

    result = {
        "neural_weight": weight,
        **retrieval_scores(rankings, dev_queries),
    }

    results.append(result)
    print(result)

hybrid_best = max(
    results,
    key=lambda row: row["f1"],
)

NEURAL_WEIGHT = hybrid_best["neural_weight"]

print("Selected hybrid result:", hybrid_best)

shortlists = {
    "dev": fuse_rankings(
        dev_neural,
        lexical["dev"],
        NEURAL_WEIGHT,
    )
}

train_neural = neural_rankings(
    retriever,
    "train",
)

shortlists["train"] = fuse_rankings(
    train_neural,
    lexical["train"],
    NEURAL_WEIGHT,
)

# The retriever stays fixed during reranker training.
retriever.cpu()

if use_amp:
    torch.cuda.empty_cache()

assert RERANK_TOP_K is None or RERANK_TOP_K >= 7


for k in [30, 50, 100, None]:
    hits = 0
    gold_total = 0
    possible_hits = 0

    for query_id, row in dev_queries.items():
        shortlist = shortlists["dev"][query_id][:k]

        available = len(
            set(shortlist)
            & row["relevant_candidates"]
        )

        hits += available
        gold_total += len(row["relevant_candidates"])
        possible_hits += min(7, available)

    print({
        "shortlist": k or "full union",
        "recall": hits / max(gold_total, 1),
        "oracle_f1": (
            2 * possible_hits
            / (7 * len(dev_queries) + gold_total)
        ),
    })

{'neural_weight': 0.0, 'precision': 0.3789346246973366, 'recall': 0.32168550873586843, 'f1': 0.3479710950528071}
{'neural_weight': 0.25, 'precision': 0.3765133171912833, 'recall': 0.31963001027749227, 'f1': 0.34574763757643134}
{'neural_weight': 0.5, 'precision': 0.3474576271186441, 'recall': 0.2949640287769784, 'f1': 0.31906614785992216}
{'neural_weight': 0.75, 'precision': 0.32082324455205813, 'recall': 0.2723535457348407, 'f1': 0.2946081156197888}
{'neural_weight': 1.0, 'precision': 0.28329297820823246, 'recall': 0.24049331963001028, 'f1': 0.26014452473596444}
Selected hybrid result: {'neural_weight': 0.0, 'precision': 0.3789346246973366, 'recall': 0.32168550873586843, 'f1': 0.3479710950528071}
{'shortlist': 30, 'recall': 0.5467625899280576, 'oracle_f1': 0.4991662034463591}
{'shortlist': 50, 'recall': 0.6104830421377184, 'oracle_f1': 0.5358532518065592}
{'shortlist': 100, 'recall': 0.6998972250770812, 'oracle_f1': 0.5558643690939411}
{'shortlist': 'full union', 'recall': 0.780061664

In [13]:
passages = {}

for split in tokens:
    passages[split] = tuple(
        {
            case_id: split_tokens(
                ids,
                PASSAGE_TOKENS,
                PASSAGE_OVERLAP,
            )
            for case_id, ids in group.items()
        }
        for group in tokens[split]
    )


def token_terms(ids):
    return [str(token_id) for token_id in ids]


passage_tfidf = TfidfVectorizer(
    analyzer=token_terms,
    sublinear_tf=True,
    dtype=np.float32,
)

passage_tfidf.fit(
    part
    for group in passages["train"]
    for document_parts in group.values()
    for part in document_parts
)

passage_vectors = {}

for split in passages:
    passage_vectors[split] = tuple(
        {
            case_id: passage_tfidf.transform(parts)
            for case_id, parts in group.items()
        }
        for group in passages[split]
    )


@lru_cache(maxsize=100000)
def select_passages(split, query_id, candidate_id):
    query_matrix = passage_vectors[split][0][query_id]
    candidate_matrix = passage_vectors[split][1][candidate_id]

    scores = (
        query_matrix @ candidate_matrix.T
    ).toarray()

    selected = []

    number_of_views = min(
        PASSAGE_VIEWS,
        *scores.shape,
    )

    for _ in range(number_of_views):
        query_index, candidate_index = np.unravel_index(
            scores.argmax(),
            scores.shape,
        )

        selected.append(
            (int(query_index), int(candidate_index))
        )

        # Avoid repeating the same passage in another view.
        scores[query_index, :] = -np.inf
        scores[:, candidate_index] = -np.inf

    return tuple(selected)

def score_pairs(model, split, q, candidates, batch_size):
    sequences = []
    owners = []

    for owner, candidate_id in enumerate(candidates):
        selected = select_passages(
            split,
            q,
            candidate_id,
        )

        for qi, ci in selected:
            query_passage = passages[split][0][q][qi]

            candidate_passage = (
                passages[split][1][candidate_id][ci]
            )

            sequence = (
                [BOS_ID]
                + query_passage
                + [EOS_ID]
                + candidate_passage
                + [EOS_ID]
            )

            sequences.append(sequence)
            owners.append(owner)

    scores = []

    for start in range(0, len(sequences), batch_size):
        ids, lengths = pack_sequences(
            sequences[start:start + batch_size]
        )

        hidden = model(ids, lengths)

        scores.append(
            model.head(hidden).squeeze(-1).float()
        )

    scores = torch.cat(scores)

    owners = torch.tensor(
        owners,
        device=device,
        dtype=torch.long,
    )

    totals = scores.new_zeros(
        len(candidates)
    ).index_add(0, owners, scores)

    counts = torch.bincount(
        owners,
        minlength=len(candidates),
    )

    return totals / counts

In [14]:
def reranker_loss(model, query_ids):
    losses = []

    for query_id in query_ids:
        candidates, positive_count = sample_candidates(
            query_id,
            shortlists["train"],
        )

        scores = score_pairs(
            model,
            "train",
            query_id,
            candidates,
            PAIR_BATCH_SIZE,
        )

        positive_scores = scores[:positive_count]
        negative_scores = scores[positive_count:]

        differences = (
            negative_scores[None, :]
            - positive_scores[:, None]
        )

        losses.append(
            F.softplus(differences).mean()
        )

    return torch.stack(losses).mean()


@torch.inference_mode()
def rerank(model, split):
    model.eval()
    rankings = {}

    for step, (query_id, ids) in enumerate(
        shortlists[split].items(),
        1,
    ):
        candidates = ids[:RERANK_TOP_K]

        with amp_context():
            values = score_pairs(
                model,
                split,
                query_id,
                candidates,
                EVAL_BATCH_SIZE,
            ).cpu().tolist()

        order = sorted(
            range(len(candidates)),
            key=lambda i: (-values[i], i),
        )

        rankings[query_id] = [
            candidates[i] for i in order
        ]

        if step % 25 == 0:
            print(
                f"{split}: "
                f"{step}/{len(shortlists[split])} "
                "queries reranked",
                flush=True,
            )

    return rankings


def combine_reranker(rankings, split, weight):
    if weight == 0:
        return shortlists[split]

    original = {
        query_id: ids[:RERANK_TOP_K]
        for query_id, ids in shortlists[split].items()
    }

    return fuse_rankings(
        rankings,
        original,
        weight,
    )


def evaluate_reranker(model):
    rankings = rerank(model, "dev")
    results = []

    for weight in FUSION_WEIGHTS:
        fused = combine_reranker(
            rankings,
            "dev",
            weight,
        )

        results.append({
            "reranker_weight": weight,
            **retrieval_scores(fused, dev_queries),
        })

    return max(
        results,
        key=lambda row: row["f1"],
    )


reranker = Phase7Model("reranker").to(device)

baseline = {
    "reranker_weight": 0.0,
    **retrieval_scores(
        shortlists["dev"],
        dev_queries,
    ),
}

reranker_history = train_stage(
    reranker,
    reranker_loss,
    evaluate_reranker,
    RERANKER_EPOCHS,
    "best_reranker.pt",
    baseline=baseline,
)

reranker | Epoch 1 | Batch 100/103 | Loss: 0.6227
reranker | Epoch 1 | Batch 103/103 | Loss: 0.7639
dev: 25/118 queries reranked
dev: 50/118 queries reranked
dev: 75/118 queries reranked
dev: 100/118 queries reranked
{'epoch': 1, 'train_loss': 0.6685836656586777, 'reranker_weight': 0.0, 'precision': 0.3789346246973366, 'recall': 0.32168550873586843, 'f1': 0.3479710950528071}
reranker | Epoch 2 | Batch 100/103 | Loss: 0.6388
reranker | Epoch 2 | Batch 103/103 | Loss: 0.5358
dev: 25/118 queries reranked
dev: 50/118 queries reranked
dev: 75/118 queries reranked
dev: 100/118 queries reranked
{'epoch': 2, 'train_loss': 0.6414668325840917, 'reranker_weight': 0.0, 'precision': 0.3789346246973366, 'recall': 0.32168550873586843, 'f1': 0.3479710950528071}


In [15]:
selected = torch.load(
    OUTPUT_DIR / "best_reranker.pt",
    map_location="cpu",
    weights_only=True,
)

print(
    "Selected reranker epoch:",
    selected["epoch"],
)

print(
    "Selected Dev result:",
    selected["result"],
)

RERANKER_WEIGHT = selected["result"]["reranker_weight"]

del reranker
gc.collect()

if use_amp:
    torch.cuda.empty_cache()


# Generate test rankings with the best retriever.
retriever.to(device).eval()

test_neural = neural_rankings(
    retriever,
    "test",
)

lexical["test"] = lexical_rankings("test")

shortlists["test"] = fuse_rankings(
    test_neural,
    lexical["test"],
    NEURAL_WEIGHT,
)

retriever.cpu()

if use_amp:
    torch.cuda.empty_cache()


if RERANKER_WEIGHT == 0:
    print(
        "Using hybrid baseline: reranking did not "
        "improve validation F1."
    )

    final_rankings = shortlists["test"]

else:
    reranker, _ = load_phase7(
        OUTPUT_DIR / "best_reranker.pt"
    )

    reranked = rerank(
        reranker,
        "test",
    )

    final_rankings = combine_reranker(
        reranked,
        "test",
        RERANKER_WEIGHT,
    )


predictions = {
    query_id: ids[:7]
    for query_id, ids in final_rankings.items()
}

assert set(predictions) == set(test_queries)

for ids in predictions.values():
    assert len(ids) == len(set(ids)) == 7
    assert set(ids) <= set(test_candidates)


prediction_path = OUTPUT_DIR / "prediction.json"

prediction_path.write_text(
    json.dumps(predictions, indent=2),
    encoding="utf-8",
)

submission_path = OUTPUT_DIR / "submission.zip"

with zipfile.ZipFile(
    submission_path,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname="prediction.json",
    )


shutil.copy2(
    MODEL_DIR / "tokenizer.json",
    OUTPUT_DIR / "tokenizer.json",
)

settings = {
    "base_model": str(MODEL_DIR / "model.pt"),
    "seed": SEED,
    "neural_weight": NEURAL_WEIGHT,
    "reranker_weight": RERANKER_WEIGHT,
    "retrieve_top_k": RETRIEVE_TOP_K,
    "rerank_top_k": RERANK_TOP_K,
    "rrf_k": RRF_K,
    "retrieval_length": RETRIEVAL_LENGTH,
    "max_chunks": MAX_CHUNKS,
    "chunk_overlap": CHUNK_OVERLAP,
    "pair_length": PAIR_LENGTH,
    "passage_tokens": PASSAGE_TOKENS,
    "passage_overlap": PASSAGE_OVERLAP,
    "passage_views": PASSAGE_VIEWS,
}

(OUTPUT_DIR / "settings.json").write_text(
    json.dumps(settings, indent=2)
)

print("Submission:", submission_path)

Selected reranker epoch: 0
Selected Dev result: {'reranker_weight': 0.0, 'precision': 0.3789346246973366, 'recall': 0.32168550873586843, 'f1': 0.3479710950528071}
Using hybrid baseline: reranking did not improve validation F1.
Submission: /kaggle/working/phase7_model3_lora/submission.zip
